In [17]:
import re
import pandas as pd

In [ ]:
import pandas as pd

cities_df = pd.read_csv("..\\info\\cities.csv")
c_aliases_df = pd.read_csv("..\\info\\c_aliases.csv")
cities_df['city_id'] = cities_df['city_id'].astype(int)
c_aliases_df['city_id'] = c_aliases_df['city_id'].astype(int)

city_lookup = cities_df.dropna(subset=['city_id', 'city_name']).set_index('city_id')['city_name'].to_dict()

alias_groups = c_aliases_df.groupby('city_id')['alias'].apply(list).to_dict()

final_list = []
for city_id, main_name in city_lookup.items():
    # Get aliases for this ID, default to empty list if none found
    aliases = alias_groups.get(city_id, [])
    
    # Create the sublist: [Main Name, Alias 1, Alias 2...]
    final_list.append([main_name] + aliases)

# Result check
print(final_list)

[['Frankfurt am Main', 'Frankfurt a.M.', 'Frankfurt (Main)'], ['Wiesbaden'], ['Kassel'], ['Darmstadt'], ['Offenbach am Main', 'Offenbach'], ['Hanau']]


In [ ]:
# Took 1m 57.9s
def process_city_stations(stops_df: pd.DataFrame, city_list: list[list[str]]):
    """
    Filters GTFS stops for stations (type 1) belonging to cities in city_list,
    mapping them to their child stops (type 0).
    """
    parent_rows = []
    mapping_rows = []

    for index, city_group in enumerate(city_list):
        # The first element is the main city name; we use the list index as city_id
        city_name = city_group[0]
        city_id = index + 1
        
        # Create regex pattern from all names in the sublist
        pattern = '|'.join([re.escape(name) for name in city_group])
        
        # Find all Type 1 (Stations) matching the city/aliases
        stations = stops_df[
            (stops_df['location_type'] == "1") & 
            (stops_df['stop_name'].str.contains(pattern, case=False, na=False))
        ].copy()

        for _, station in stations.iterrows():
            s_id = station['stop_id']
            stop_name = station['stop_name']
            lat, lon = station['stop_lat'], station['stop_lon']

            # 1. Populate parent_stations data
            parent_rows.append({
                'stop_id': s_id,
                'stop_name': stop_name,
                'city_id': city_id,
                'stop_lat': lat,
                'stop_lon': lon
            })

            # 2. Find child stops (Type 0) where parent_station matches current s_id
            children = stops_df[stops_df['parent_station'] == s_id]
            
            for _, child in children.iterrows():
                mapping_rows.append({
                    'stop_id': child['stop_id'],
                    'station_id': s_id,
                    'stop_lat': child['stop_lat'],
                    'stop_lon': child['stop_lon']
                })

    # Convert to DataFrames
    df_parents = pd.DataFrame(parent_rows)
    df_mappings = pd.DataFrame(mapping_rows)

    return df_parents, df_mappings

stops_df = pd.read_csv("..\\latest\\stops.txt", dtype="string")

parents_df, mappings_df = process_city_stations(stops_df, final_list)

# Export to CSV
parents_df.to_csv("parent_stations.csv", index=False)
mappings_df.to_csv("stop_to_station.csv", index=False)

print(f"Processed {len(parents_df)} parent stations across all cities.")

Processed 2346 parent stations across all cities.


In [ ]:
# AI GEN
def get_station_route_types(latest_path: str, stop_to_station_df: pd.DataFrame):
    # 1. Load the necessary GTFS files
    stop_times = pd.read_csv(f"{latest_path}/stop_times.txt", usecols=['trip_id', 'stop_id'], dtype=str)
    trips = pd.read_csv(f"{latest_path}/trips.txt", usecols=['trip_id', 'route_id'], dtype=str)
    routes = pd.read_csv(f"{latest_path}/routes.txt", usecols=['route_id', 'route_type'], dtype=str)

    # 2. Join tables to connect stop_id to route_type
    # stop_times + trips on trip_id
    st_trips = pd.merge(stop_times, trips, on='trip_id')
    # + routes on route_id
    st_routes = pd.merge(st_trips, routes, on='route_id')

    # 3. Get unique combinations of stop_id and route_type
    stop_route_mapping = st_routes[['stop_id', 'route_type']].drop_duplicates()

    # 4. Map these child stops to their Parent Stations
    # Use the stop_to_station_df created in the previous step
    station_route_mapping = pd.merge(
        stop_route_mapping, 
        stop_to_station_df[['stop_id', 'station_id']], 
        on='stop_id'
    )

    # 5. Create the "stop_route_type" table (Mapping child stops to types)
    stop_route_type = stop_route_mapping.rename(columns={'route_type': 'route_type_id'})

    # 6. Create the "station_route_type" table (Mapping parents to types)
    # A station might have multiple platforms (stops) with different route types
    station_route_type = station_route_mapping[['station_id', 'route_type']].drop_duplicates()
    station_route_type = station_route_type.rename(columns={'route_type': 'route_type_id'})

    return stop_route_type, station_route_type

# --- Implementation ---
latest_dir = "../latest"
# mappings_df comes from your previous process_city_stations function
stop_route_df, station_route_df = get_station_route_types(latest_dir, mappings_df)

# Save the results
stop_route_df.to_csv("stop_route_type.csv", index=False)
station_route_df.to_csv("station_route_type.csv", index=False)